# EDA Report: Indonesian Food Image Dataset

Notebook ini menyajikan exploratory data analysis (EDA) untuk dataset klasifikasi makanan Indonesia yang digunakan pada Forum 04. Fokus analisis meliputi struktur split dataset, distribusi jumlah gambar per kelas, variasi ukuran gambar, rasio aspek, ukuran file, dan sampel visual tiap kategori.

In [1]:
%matplotlib inline

import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

warnings.filterwarnings("ignore")
sns.set_theme(style = "whitegrid", context = "notebook")
plt.style.use("seaborn-v0_8")
pd.set_option("display.max_columns", None)

SEED = 42
np.random.seed(SEED)

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 1. Dataset Source and Local Cache Strategy

Dataset utama berasal dari Kaggle: `rizkyyk/dataset-food-classification`. Notebook ini memakai strategi local caching: jika folder `dataset/` sudah berisi gambar, analisis memakai cache lokal; jika belum, notebook mencoba mengunduh dataset dari Kaggle menggunakan `kagglehub`.

In [ ]:
import shutil

import kagglehub

def resolve_project_root() -> Path:
    current_dir = Path.cwd().resolve()
    candidate_dirs = [current_dir, current_dir.parent]
    required_markers = {
        "EDA_Report_Indonesian_Food.ipynb",
        "Forum04-requirements.txt",
    }

    for each_candidate in candidate_dirs:
        if all((each_candidate / each_marker).exists() for each_marker in required_markers):
            return each_candidate

    raise FileNotFoundError(
        "Forum04 project root was not found. Please run this notebook from the forum04 workspace folder."
    )

DATASET_HANDLE = "rizkyyk/dataset-food-classification"
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}
project_root = resolve_project_root()
dataset_folder = project_root / "dataset"
dataset_folder.mkdir(parents = True, exist_ok = True)

def has_local_images(folder: Path) -> bool:
    return any(
        each_file.is_file() and each_file.suffix.lower() in VALID_EXTENSIONS
        for each_file in folder.glob("**/*")
    )

def kaggle_credentials_exist() -> bool:
    return any([
        bool(os.getenv("KAGGLE_API_TOKEN")),
        (Path.home() / ".kaggle" / "kaggle.json").exists(),
        (Path.home() / ".kaggle" / "access_token").exists(),
    ])

print(f"Project root resolved to: {project_root}")
print(f"Dataset folder resolved to: {dataset_folder}")

if has_local_images(dataset_folder):
    print("Using local dataset cache from dataset/.")
elif kaggle_credentials_exist():
    download_root = Path(kagglehub.dataset_download(handle = DATASET_HANDLE))
    candidate_sources = [
        download_root / "dataset_gambar",
        download_root / "dataset",
        download_root,
    ]
    source_folder = next((item for item in candidate_sources if item.exists()), None)

    if source_folder is None:
        raise FileNotFoundError("Downloaded dataset structure was not recognized.")

    shutil.copytree(src = source_folder, dst = dataset_folder, dirs_exist_ok = True)
    print("Dataset downloaded and copied into dataset/.")
else:
    raise RuntimeError(
        "No local dataset found and Kaggle credentials are not configured. "
        "Set KAGGLE_API_TOKEN or place kaggle.json under ~/.kaggle/."
    )

In [ ]:
def resolve_split_directory(base_folder: Path, candidates: list[str]) -> Path:
    for each_candidate in candidates:
        candidate_path = base_folder / each_candidate
        if candidate_path.exists():
            return candidate_path

    raise FileNotFoundError(f"Could not find split directory from candidates: {candidates}")

split_aliases = {
    "train": ["train"],
    "val": ["valid", "val", "validation"],
    "test": ["test"],
}

split_directories = {
    split_name: resolve_split_directory(dataset_folder, aliases)
    for split_name, aliases in split_aliases.items()
}

records = []
for split_name, split_path in split_directories.items():
    for each_image_path in split_path.glob("**/*"):
        if each_image_path.is_file() and each_image_path.suffix.lower() in VALID_EXTENSIONS:
            records.append({
                "split": split_name,
                "label_raw": each_image_path.parent.name,
                "image_path": str(each_image_path),
                "extension": each_image_path.suffix.lower(),
                "file_size_kb": each_image_path.stat().st_size / 1024,
            })

image_df = pd.DataFrame(records)
image_df.head()

## 2. Dataset Inventory

Bagian ini memeriksa jumlah gambar total, jumlah kelas, distribusi split, dan komposisi label. Analisis ini penting karena ketimpangan kelas dapat memengaruhi stabilitas training CNN.

In [ ]:
dataset_summary = pd.DataFrame({
    "metric": [
        "total_images",
        "total_classes",
        "available_splits",
        "file_extensions",
    ],
    "value": [
        len(image_df),
        image_df["label_raw"].nunique(),
        ", ".join(sorted(image_df["split"].unique())),
        ", ".join(sorted(image_df["extension"].unique())),
    ],
})

display(dataset_summary)
display(image_df.groupby(["split", "label_raw"]).size().rename("image_count").reset_index().head(10))

fig, axes = plt.subplots(nrows = 1, ncols = 2, figsize = (18, 5))

split_counts = image_df.groupby("split").size().sort_values(ascending = False)
axes[0].bar(split_counts.index, split_counts.values, color = ["#2E86AB", "#F18F01", "#C73E1D"])
axes[0].set_title("Image Count by Split")
axes[0].set_xlabel("Split")
axes[0].set_ylabel("Image Count")

label_counts = image_df.groupby("label_raw").size().sort_values(ascending = False)
axes[1].bar(label_counts.index, label_counts.values, color = "#54A24B")
axes[1].set_title("Image Count by Food Category")
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Image Count")
axes[1].tick_params(axis = "x", rotation = 90)

plt.tight_layout()
plt.show()

## 3. Image Geometry Analysis

Ukuran gambar yang tidak seragam merupakan temuan penting karena CNN memerlukan input tensor dengan dimensi yang konsisten. Oleh sebab itu, kita ukur lebar, tinggi, dan rasio aspek dari seluruh gambar sebelum preprocessing.

In [ ]:
geometry_records = []
for each_row in image_df.itertuples(index = False):
    with Image.open(each_row.image_path) as img:
        width, height = img.size

    geometry_records.append({
        "split": each_row.split,
        "label_raw": each_row.label_raw,
        "width": width,
        "height": height,
        "aspect_ratio": width / height,
        "file_size_kb": each_row.file_size_kb,
    })

geometry_df = pd.DataFrame(geometry_records)
display(geometry_df.describe().T)

fig, axes = plt.subplots(nrows = 2, ncols = 2, figsize = (16, 10))
sns.histplot(data = geometry_df, x = "width", hue = "split", bins = 25, ax = axes[0, 0])
axes[0, 0].set_title("Width Distribution")

sns.histplot(data = geometry_df, x = "height", hue = "split", bins = 25, ax = axes[0, 1])
axes[0, 1].set_title("Height Distribution")

sns.histplot(data = geometry_df, x = "aspect_ratio", hue = "split", bins = 25, ax = axes[1, 0])
axes[1, 0].set_title("Aspect Ratio Distribution")

sns.histplot(data = geometry_df, x = "file_size_kb", hue = "split", bins = 25, ax = axes[1, 1])
axes[1, 1].set_title("File Size Distribution (KB)")

plt.tight_layout()
plt.show()

## 4. Sample Gallery

Visual inspection membantu memastikan bahwa setiap folder memang mewakili kelas yang benar dan bahwa gambar tidak rusak atau tertukar. Pada galeri di bawah, notebook menampilkan satu contoh gambar dari setiap kategori.

In [ ]:
sample_df = image_df.groupby("label_raw", as_index = False).first().sort_values(by = "label_raw")
num_samples = len(sample_df)
num_cols = 3
num_rows = int(np.ceil(num_samples / num_cols))

fig, axes = plt.subplots(num_rows, num_cols, figsize = (15, 5 * num_rows))
axes = np.array(axes).reshape(-1)

for each_axis, each_row in zip(axes, sample_df.itertuples(index = False)):
    with Image.open(each_row.image_path) as img:
        each_axis.imshow(img.convert("RGB"))
    each_axis.set_title(each_row.label_raw)
    each_axis.axis("off")

for each_axis in axes[num_samples:]:
    each_axis.axis("off")

plt.tight_layout()
plt.show()

## 5. Data Quality Notes

Interpretasi umum yang biasanya diambil dari EDA ini adalah: 

1. Jika distribusi label tidak seimbang, maka training berisiko bias ke kelas mayoritas.
2. Jika ukuran gambar sangat bervariasi, preprocessing ke ukuran tetap `224 x 224` menjadi kebutuhan, bukan pilihan.
3. Jika rasio aspek sangat tersebar, resize lalu center crop dapat membuang konteks pinggir gambar sehingga augmentasi bisa menjadi tahap lanjutan yang relevan.
4. Jika terdapat gambar dengan ukuran file ekstrem, data cleaning tambahan mungkin diperlukan.